# 04 · anatomy: registration and the Allen atlas

So far every pixel has been a little spectrum floating in space. We know its lipids, we know
which section it came from, we even know where it sits on the slide. What we still do not know
is the thing a neuroscientist asks first: where in the brain is it? Is this pixel in the
caudoputamen or the hippocampus, in grey matter or in a white-matter tract, in cortical layer 1
or layer 6? Without that label a lipid map is a pretty picture. With it, the map becomes
anatomy, and anatomy is what lets us compare control against pregnant region by region, and
later join our lipids to gene expression in the exact same regions.

This notebook is about how every pixel gets an anatomical address. The procedure is called
registration to a reference atlas, and the reference is the Allen Mouse Brain Common Coordinate
Framework, version 3 (CCFv3). We will:

1. understand what a reference atlas is, and why we need a per-pixel anatomical coordinate,
2. understand registration as an affine transform plus a nonlinear warp, built from a little
   linear algebra you can hold in your head,
3. meet ABBA, the community-standard tool, as a concept: it is the one piece of infrastructure
   we ship pre-computed rather than run live,
4. load the per-pixel CCF coordinates and region labels that already live on our two sections,
5. make the coordinate-to-region lookup completely explicit, and read the slicing angle off the data,
6. draw region maps for control and pregnant, split grey from white matter, and check that a
   myelin lipid traces the white-matter tracts on its own,
7. build region-level lipid views, the mean lipid profile of every Allen region, which is the
   table that powers the region-level differential test and the gene join in later notebooks.

The whole notebook runs on the two sections you already normalized in notebook 3: a control
female and a pregnant female, cut at the same coronal plane near AP 6.5.

In [ ]:
# the stack you know, plus our course helper package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

import cajal_lipidomics as cl
from cajal_lipidomics import plotting, analysis
from cajal_lipidomics.style import set_style, FS
set_style()  # the course-wide clean figure style

# one global seed so every number and figure below is reproducible
RNG_SEED = 0
np.random.seed(RNG_SEED)

print("ready. cajal_lipidomics", cl.__version__)

check: you should see `ready. cajal_lipidomics 0.0.1` and no red error. If you get a
`ModuleNotFoundError`, your notebook is on the wrong kernel; pick the `cajal-lipidomics` kernel
from the kernel picker.

## 1. what a reference atlas is, and why we need one

A pixel is one MALDI measurement. The laser samples the tissue on a regular grid spaced about 25
micrometers apart, so each pixel stands for a little 25-micrometer tile of brain (the ablation spot
itself is only a few microns wide). The mass spectrometer hands us,
for each pixel, an intensity for every lipid. It does not hand us a brain region. The scanner
only knows that this spot was at row `x`, column `y` of the slide. That slide coordinate is
meaningless across sections: the tissue was placed at a slightly different angle, cut at a
slightly different depth, the brain was a little bigger or smaller. Two pixels with the same
`(x, y)` on two slides are almost never the same piece of brain.

We need a shared coordinate system that every section can be mapped into, so that "this location"
means the same thing in control and in pregnant, in your brain and in mine. That shared system is
a reference atlas.

The Allen Mouse Brain Common Coordinate Framework v3 (CCFv3) is the standard one. Think of it as
two things bolted together:

- a 3D coordinate space. The whole mouse brain is described as a box of cubic voxels, each 25
  micrometers on a side. A point in the brain is just three numbers, `(x, y, z)`, in millimeters,
  measuring how far along the three anatomical axes you are: the anterior-posterior (AP) axis
  (front to back), the dorsal-ventral axis (top to bottom), and the medial-lateral axis (midline
  to side). This is the same idea as latitude, longitude, and altitude on Earth: any place gets
  one fixed triple of numbers.
- an annotation volume. For every one of those voxels, the Allen team filled in which brain region
  it belongs to. The brain was carved, by experts, into a tree of about a thousand structures: big
  divisions like cortex, striatum, thalamus, then finer ones like caudoputamen, then finer still
  like cortical layers. Each region has a short acronym (`CP` for caudoputamen) and a color chosen
  so that related regions look alike.

So once a pixel has a CCF coordinate `(x, y, z)`, getting its region is a pure lookup: go to that
voxel in the annotation volume, read off the region. Coordinate first, anatomy second. The hard
part, and the subject of this notebook, is getting that coordinate. That is registration: bending
each real, distorted section until it lines up with the atlas, then reading each pixel's atlas
coordinate.

## 2. registration as an affine transform plus a nonlinear warp

Registration is the act of finding the geometric transformation that takes your section onto the
atlas. It comes in two layers, and the layers matter, so we build the intuition from the linear
algebra up.

### the affine layer: rotate, scale, shear, translate

The first layer handles the gross misalignment: your section is rotated a few degrees, slightly
bigger or smaller than the atlas, maybe squashed along one axis because the tissue shrank, and
shifted off-center. Every one of those is an affine transformation: a linear map (a 2x2 matrix `A`
in 2D) followed by a translation (a vector `t`). A point `p` moves to

$$ p' = A\,p + t. $$

The matrix `A` packs four operations into four numbers: rotation, isotropic scaling (same in both
directions), anisotropic scaling (different per axis, which is how you undo a 20% shrinkage along
one direction), and shear. The translation `t` slides the whole thing. Affine maps keep straight
lines straight and parallel lines parallel: a grid drawn on the section stays a grid, just rotated,
stretched, and sheared. That is the key limitation, and why we need a second layer.

🔬 **TASK.** Build the intuition with a tiny demo. We take a small square grid of points, apply one
affine transform (a rotation, an anisotropic scale, and a shift), and look at what happened. No
atlas yet, just the geometry.

In [ ]:
# a small grid of points: this stands in for "landmarks on a section"
# 🔬 your code here


check: the red grid is a rotated, stretched, shifted copy of the gray grid, but it is still a
perfect grid: straight lines stayed straight, parallel stayed parallel. That is exactly what an
affine map can and cannot do.

❓ **QUESTION.** Real tissue does not just rotate and stretch uniformly. A section gets torn, folded,
locally swollen near a ventricle, compressed where the blade dragged. Can a single matrix `A` undo a
local fold in one corner while leaving the rest untouched? No: one matrix acts on the whole plane
identically. That is the job of the second layer.

### the nonlinear layer: a smooth warp that bends locally

After the affine has done the gross alignment, regions are roughly in place but the fine boundaries
still do not match: the hippocampus is a touch too curved here, the cortex a bit too thick there.
The second layer is a nonlinear warp, a smooth deformation field that can push each point a different
little amount, bending locally without tearing.

Formally it is a displacement field: at every location it stores a small arrow saying "move this bit
of tissue by this much, in this direction". The good algorithms constrain the field to be smooth and
invertible (a diffeomorphism), so tissue never folds over itself or rips. Methods like elastix splines
(used by ABBA) or LDDMM (used by STalign) learn this field by minimizing the mismatch between the
warped section and the atlas, while penalizing roughness.

🔬 **TASK.** Add a gentle nonlinear warp on top of the affine grid, so you can see the difference. We
push each point by a smooth sine-shaped displacement: now the lines bend.

In [ ]:
# take the affine-transformed grid and add a smooth, local, nonlinear displacement
# 🔬 your code here


check: the blue points no longer form straight rows. The gray arrows show each point being
nudged a little, differently in different places, but smoothly. That bending is what lets the
section's curved hippocampus snap onto the atlas's curved hippocampus.

So the full registration is: affine first to get the gross fit, nonlinear warp second to bend the
fine boundaries into place. Run it, and you can read off, for every pixel, the atlas coordinate it
landed on. That coordinate is the CCF triple `(xccf, yccf, zccf)` we will load shortly.

## 3. ABBA: the one piece we ship pre-computed

In a real lab you do not write the warp by hand. You use a tool. The community standard for
registering serial brain sections to the Allen CCFv3 is ABBA, which stands for Aligning Big Brains
and Atlases, from Nicolas Chiaruttini's group at the EPFL bioimaging platform.

ABBA is a Fiji (ImageJ) plugin, usually driven together with QuPath, through a mouse-driven
graphical interface. You feed it your section images, place them roughly along the atlas, and it runs
exactly the two layers we just built:

- an optional machine-learning pre-alignment (DeepSlice) guesses which atlas slice you are looking at
  and the cutting angle,
- an affine elastix step does the gross rotate/scale/shear/translate fit,
- a spline elastix step does the nonlinear warp,
- and for stubborn slices you hand-place landmark pairs in a tool called BigWarp.

A crucial detail for our data: ABBA also corrects the slicing angle. A coronal block is almost never
cut perfectly perpendicular to the AP axis, so a single physical section actually grazes a small
range of AP levels across its width. ABBA estimates that tilt and accounts for it. You will see the
fingerprint of this in our pregnant section in a moment.

When ABBA is done, you export a per-pixel CCF coordinate map: an image whose channels are the `x`,
`y`, `z` atlas coordinates of every pixel. Run that coordinate through the Allen annotation volume and
each pixel also gets a region acronym and color. That pair of products, the coordinate and the region
label, is exactly what we hand you on the two sections.

### why we ship this one ready-made

Everywhere else in this course you build the analysis yourself, from raw METASPACE ions through
normalization, embedding, clustering, and differential testing, with the code open in front of you.
Registration is the single exception, and it is worth understanding why:

- ABBA's core (elastix, BigWarp, DeepSlice) is Java and C++, so it cannot be unrolled into transparent
  Python the way every other step can. 
- It is a desktop GUI. Live-driving a mouse-heavy interface for two sections is high effort and low
  teaching payoff, and its headless mode on Linux is unreliable.
- Registration is a one-time setup, not a method you need to master to do the science. The differential
  lipidomics and the gene join downstream are where the value is. You only need a paranoid eye to do registration, and for that you can count on online tutorials: not much science is involved.

So here ABBA is a concept with a described demo: in your own lab you would create a QuPath project, run
DeepSlice then affine then spline elastix, fix slices with BigWarp, and export the coordinate map. The
per-pixel CCF coordinate and region label that you load next are that export. They are the only thing in
this entire course that arrives pre-computed.

## 4. loading the per-pixel CCF coordinates

Time to look at the real output. We pick up exactly where notebook 3 left off, loading the normalized
two-section pair `03_normalized.h5ad`. Everything registration produced is already sitting in
`adata.obs`: the CCF coordinates and, from the Allen lookup, the region acronym and color for every
pixel.

One housekeeping step. This file keeps two versions of the data: `adata.X` holds the raw per-pixel ion
images, and `adata.layers["umaia"]` holds the uMAIA-normalized values you computed in notebook 3. For
the lipid views in this notebook we want the normalized values, so we promote that layer to `adata.X`
once, up front, and everything downstream uses it.

🔬 **TASK.** Load the substrate, switch to the normalized values, and confirm its shape.

In [ ]:
# pick up notebook 3's output: control + pregnant, same coronal plane (~AP 6.5)
adata = ad.read_h5ad("../../data/derived/03_normalized.h5ad")
adata.X = adata.layers["umaia"]      # use the uMAIA-normalized values from notebook 3

print("pixels x lipids:", adata.shape)
print("\nconditions:")
print(adata.obs["Condition"].value_counts())
print("\nsections (SectionID):")
print(adata.obs["SectionID"].value_counts())

The anatomy lives in a handful of `obs` columns. The provided registration gives us, per pixel,
the CCF coordinate and the region label that the annotation-volume lookup returned. Let us look at
exactly those columns for a few pixels.

🔬 **TASK.** Print the CCF coordinates and the Allen region acronym and color for the first few pixels.

In [ ]:
# the anatomy columns produced by registration + Allen lookup
# 🔬 your code here


Read one row. `xccf, yccf, zccf` are the pixel's position in the Allen brain, in millimeters
along the three anatomical axes. `acronym` is the region's short code, and `allencolor` is the Allen
palette color for that region as a hex string. The coordinate is what registration computed directly;
the acronym and color are what fell out of looking that coordinate up in the annotation volume.

❓ **QUESTION.** There are three CCF numbers but each pixel only sits on a 2D section. Where do all
three come from? The two in-plane coordinates come from where the pixel landed after the warp. The
out-of-plane one (here `xccf`, the AP axis) comes from which atlas level the section was placed at,
plus the slicing-angle correction. Together they pin the pixel in 3D.

## 5. how a coordinate indexes the annotation volume

The acronym and color came from a lookup, and it is worth making that lookup completely explicit,
because once you have seen it there is nothing mysterious left. The CCF is a grid of 25-micrometer
voxels. A millimeter is 1000 micrometers, so one millimeter spans 1000 / 25 = 40 voxels. To turn a
coordinate in millimeters into a voxel index you multiply by 40 and take the floor (drop the
fractional part), because a coordinate anywhere inside a voxel belongs to that voxel:

$$ \text{index} = \big\lfloor \text{ccf}_\text{mm} \times 40 \big\rfloor. $$

That integer triple is the address you read in the Allen annotation volume to get the region. We do
not ship the multi-gigabyte annotation volume here (that is one reason the lookup was done for you up
front), but we can still reconstruct the voxel indices from the coordinates, to see the first half of
the chain with our own eyes.

🔬 **TASK.** Convert the CCF coordinates into integer voxel indices.

In [ ]:
# index = floor(ccf_mm * 40), because 1 mm = 40 voxels at 25 um
# 🔬 your code here


So the full chain is: registration gives `(xccf, yccf, zccf)` in mm, multiply by 40 and floor to
get an integer voxel index, look that voxel up in the Allen annotation volume, and out comes the region
`acronym` and its `allencolor`. The file you loaded already carries the end of that chain; the cell
above reconstructed its first step.

Now the slicing-angle point from section 3, made concrete. A perfectly coronal section sits at one
single AP level, so its `x_index` (the AP voxel) should take exactly one value. A section cut at a
slight tilt grazes a small range of AP levels across its width, so its `x_index` spreads over several
values. Let us check our two sections.

🔬 **TASK.** Count how many distinct AP levels each section spans, and the millimeter range of its AP
coordinate.

In [ ]:
# AP axis is x here: a flat coronal cut -> one x_index; a tilted cut -> a small range
# 🔬 your code here


check: the control section sits at a single AP level (one `x_index`, a flat coronal plane), while
the pregnant section spreads across a few dozen AP levels over roughly a millimeter. That spread is the
slicing-angle correction doing its job: ABBA recognized that the pregnant block was cut at a small tilt
and assigned each part of the section to the AP level it actually belongs to, rather than forcing the
whole thing onto one plane. Both sections still center on the same plane near AP 6.5, so they are
genuinely comparable; the tilt is just honestly accounted for.

## 6. region maps: control vs pregnant

Now the payoff. Every pixel carries an Allen color in `allencolor`, so coloring pixels by it draws the
brain regions directly onto the section, in the standard Allen palette. We use the helper
`cl.plotting.spatial_categorical`, which scatters each pixel at its CCF position `(zccf, -yccf)` and
paints it with the stored hex color. (We plot `-yccf` so that dorsal is up, the way you would view a
coronal section.)

Before calling the helper, see what it does in one line: it is just a scatter where the color of each
point is read straight from a column of hex strings. No colormap, no normalization, because the colors
are already chosen by the Allen atlas.

💡 **HINT.** Open `src/cajal_lipidomics/plotting.py` and read `spatial_categorical` before you
run it: it is a thin per-section scatter that reads each point's color straight from your hex
column, nothing more. Every helper in this notebook is short enough to read in a minute, and
reading it is the difference between running code and understanding it.

🔬 **TASK.** Draw the Allen region map for both sections side by side.

In [ ]:
# region map: each pixel painted with its Allen region color (allencolor)
# 🔬 your code here


check: two coronal sections, each carved into colored territories. The big central blob is the
caudoputamen, the folded outer ribbon is cortex, the paired structures below are hippocampus and
thalamus. The two sections show the same anatomy in the same colors, because both were registered into
the same atlas. That shared frame is exactly what lets us compare them.

❓ **QUESTION.** The two sections are not pixel-for-pixel identical: tissue is torn here, missing there,
a little rotated. Yet the colored regions land in the same places. What made that possible? The
registration: each raw, distorted section was warped onto the common atlas, so anatomy, not slide
position, decides a pixel's color.

Let us confirm how many distinct Allen regions we are working with, and which are the largest.

In [ ]:
# how many Allen regions appear, and the most-sampled ones
print("distinct Allen regions on these two sections:", adata.obs["acronym"].nunique())
print("\nlargest regions by pixel count:")
print(adata.obs["acronym"].value_counts().head(10))

So our two sections together touch 174 distinct Allen regions. The caudoputamen (`CP`) is the
biggest, then piriform cortex (`PIR`), then medial amygdala and a hippocampal field, exactly the
structures you expect at this coronal level. Each of these is now a group we can average lipids over.

### the outline view: region boundaries as contours

A filled region map is honest but busy: 174 colors fight for attention, and most of the time
the thing you actually want is just the outline, the line where one region ends and the next
begins, drawn thin over a lipid map so you can read chemistry and anatomy in the same glance.
That outline is a contour, and you can build it from nothing but the region labels you already
carry.

The rule is local and almost trivially simple: a pixel sits on a contour when at least one of
its neighbours belongs to a different region. Pixels deep inside a region, surrounded by their
own kind, are not edges; only the pixels straddling a boundary are. Rasterize the region ids
back onto the pixel grid, compare each pixel to its right and lower neighbour, and keep the ones
that disagree. That is the whole algorithm, and it is exactly what `cl.plotting.allen_contours`
does.

💡 **HINT.** Open `src/cajal_lipidomics/plotting.py` and read `allen_contours` before you call
it. It is about ten lines: it lays the region codes on a grid, marks where a neighbour differs,
and scatters those edge pixels. Once you have read it the contour stops being magic.

🔬 **TASK.** First the idea on a toy grid: a 6x6 field carved into three regions by hand, with
the boundary pixels found by the neighbour-differs rule and circled.

In [ ]:
# a tiny region map: a 6x6 grid carved into three regions by hand
# 🔬 your code here


check: every circled pixel touches a different-colored neighbour; every uncircled pixel is deep
inside one region. That is the entire definition of a contour, and `allen_contours` runs the
same two comparisons on the real section instead of on a 6x6 toy.

🔬 **TASK.** Now the real thing: take the control section, draw one lipid in pixel space, and lay
the Allen boundaries on top with `allen_contours`.

In [ ]:
# one section, one lipid, boundaries on top. PC 34:1 is a grey-matter-leaning phospholipid,
# so the cortical and striatal outlines should fall right where its intensity changes.
# 🔬 your code here


check: the black tracery lands on the seams of the intensity map, the outline of the
caudoputamen, the cortical ribbon, the hippocampal fields. The contour was built only from
region labels, the color only from a lipid, and yet the lines sit on the chemical edges:
anatomy and lipidomics agreeing again, now boundary by boundary. The same one-line call drops
these outlines onto any lipid map, which is how you orient a chemical image without burying it
under 174 filled colors.

## 7. the grey matter vs white matter split

The single biggest division in brain lipid composition is grey matter versus white matter. White matter
is dense with myelin, the lipid-rich insulating sheath that oligodendrocytes wrap around axons, and
myelin is built largely from sphingolipids like HexCer (hexosylceramides) and Cer (ceramides). Grey
matter, packed with cell bodies and synapses, is richer in glycerophospholipids. If a lipidomic method
cannot see the grey/white split, it cannot see anything; it is the first sanity check.

How do we get the split from what we have? The Allen ontology gathers every white-matter structure
under one node called fiber tracts: the corpus callosum, fimbria, internal capsule, external capsule,
and the rest. We do not ship the full ontology here, but we do not need it, because the Allen palette
already encodes that grouping in the color: every region in the fiber-tracts subtree is painted with
the same single gray, `#cccccc`. So a pixel is white matter exactly when its `allencolor` is that gray.
This is not a hand-guessed list of acronyms that could miss a tract; it is the atlas's own definition of
white matter, read straight off the color we already carry. (For contrast the ventricles get a different
gray, `#aaaaaa`, and unassigned tissue gets white, `#ffffff`, so the color cleanly separates the three.)

🔬 **TASK.** Label each pixel grey or white by its Allen color, print which fiber-tract regions the
color picked out so the selection is auditable, and count how the pixels split.

In [ ]:
# the Allen palette paints the entire fiber-tracts (white-matter) subtree with one gray, #cccccc.
# selecting on that color is the atlas's own white-matter definition, with no acronym guessing.
# 🔬 your code here


About one pixel in ten is white matter, the rest grey, the right ballpark for a coronal
section at this level (the corpus callosum body, internal capsule, fimbria, external capsule, and
other tracts cut through here). Now draw the split on the sections so you can see the tracts.

🔬 **TASK.** Map a simple two-color scheme: black for white matter, light gray for grey matter.

In [ ]:
# paint white matter black, grey matter light-gray, to expose the tracts
# 🔬 your code here


check: the black pixels trace clear bands, the internal capsule cutting through the caudoputamen,
the fimbria near the hippocampus, the external capsule wrapping the cortex. These are the white-matter
tracts, recovered purely from the Allen ontology, with no lipid information used. Next we check whether
the lipids agree.

### does myelin lipid track the white-matter anatomy?

Here is the test that ties anatomy to chemistry. If white matter is myelin and myelin is sphingolipid,
then a myelin marker lipid like HexCer 42:2 should be far brighter in the white-matter pixels than in
grey. `HexCer` is the class (hexosylceramide) and `42:2` is the sum composition (42 acyl carbons, 2
double bonds). We never told the lipids where the tracts are; if they light up the tracts anyway,
anatomy and lipidomics are telling the same story.

🔬 **TASK.** Compare the mean intensity of HexCer 42:2 in white vs grey matter, then plot the lipid
spatially so you can see it overlap the tracts.

In [ ]:
# myelin marker: HexCer 42:2. Compare its mean in white vs grey matter.
# 🔬 your code here


In [ ]:
# now SEE it: the helper draws one lipid on both sections with a shared color scale.
# var_names are chemical formulas, so address the lipid by its formula (the var index at j).
plotting.spatial_lipid(adata, adata.var_names[j], section_key="SectionID",
                       title_key="Condition", point_size=3.0,
                       background=False, show_contours=False)
plt.show()

check: HexCer 42:2 is several times brighter in white matter than in grey, and the spatial map
lights up exactly the bands you saw in black a moment ago. The myelin lipid traces the white-matter
anatomy on its own. That agreement between an anatomy label (from the Allen atlas) and a lipid signal
(from the mass spectrometer), with neither one informed by the other, is the first proof that the
registration is sound and that lipids carry real anatomical structure.

### which lipids mark white matter, ranked

One lipid lighting up the tracts is a good anecdote; the honest question is which lipids, out
of all 104, are most enriched in white matter versus the rest of the brain. That is a
one-number-per-lipid contrast: take each lipid's mean in the white-matter pixels, its mean
everywhere else, and rank lipids by the log2 of that ratio. `cl.plotting.marker_barplot` does
exactly this and draws the top of the list as a sorted horizontal bar chart, turning the
single-lipid check into a full ranking.

💡 **HINT.** Open `src/cajal_lipidomics/plotting.py` and read `marker_barplot` before calling
it. It is a few lines: log2(mean-in-mask / mean-in-rest) per lipid, sort, draw the top. Knowing
that, you can read every bar below as a fold change of white matter over everything else.

The helper labels each bar with the var name, and our `var_names` are chemical formulas, so we
hand it a copy whose `var_names` are the readable lipid names, and color the sphingolipid
classes (HexCer, SHexCer, SM, Cer) red so the myelin signal is impossible to miss.

🔬 **TASK.** Rank the lipids by white-vs-rest enrichment and read off the top of the list.

In [ ]:
# a copy addressed by lipid name (not formula), so the bars are readable
# 🔬 your code here


check: the top of the ranking is sphingolipids, in red. HexCer 42:2 sits up near the top
alongside the other HexCer, SM, and Cer species, exactly the myelin chemistry white matter is
built from. The barplot knew nothing about myelin; it only compared two pixel sets. The
single-lipid test has generalized into a whole class-level signature, and it is precisely the
one biochemistry predicts. With the grey/white split now validated lipid by lipid, we can move
from individual pixels to whole regions.

## 8. region-level lipid views: the mean lipid profile of every region

We have a region label on every pixel and a normalized intensity for every lipid. The natural summary
is the region-by-lipid matrix: for each Allen region, the mean intensity of each lipid across all its
pixels. One row per region, one column per lipid. This single table is the workhorse for everything
anatomical that follows: it is what a region-level differential test compares between conditions, and it
is the shape that matches a region-by-gene table when we join to gene expression later.

Before any helper, build it by hand in two lines, because the operation is just a `groupby` mean. Seeing
it built makes the helper obvious.

🔬 **TASK.** Construct the region-by-lipid mean matrix with a plain pandas groupby.

In [ ]:
# region x lipid: mean intensity of each lipid within each Allen region
# 🔬 your code here


That is the whole idea: 174 regions, 104 lipids, each entry a mean intensity. The helper
`cl.plotting.sorted_lipid_heatmap` does exactly this groupby mean and then draws it as a heatmap, with
one extra touch: it orders both the rows (regions) and the columns (lipids) by hierarchical clustering on
cosine similarity, so that regions with similar lipid profiles sit next to each other and the block
structure becomes visible. The sorting is the art; the table underneath is the two-line groupby you just
wrote.

💡 **HINT.** Open `src/cajal_lipidomics/plotting.py` and read `sorted_lipid_heatmap`: it runs
the same groupby mean you just wrote, 0-1 normalizes each lipid, then orders rows and columns by
hierarchical clustering on cosine similarity. The clustering is the only part you have not
already coded by hand.

🔬 **TASK.** Draw the region-by-lipid heatmap, clustered on both axes.

In [ ]:
# the helper: per-lipid 0-1 normalize, group-mean by acronym, then cosine-cluster both axes
# 🔬 your code here


check: the heatmap shows clear blocks, groups of regions (rows) that share a group of high lipids
(columns). Those blocks are the lipidomic signature of anatomy. A band of regions lighting up the
sphingolipid columns is the white matter; other blocks are cortical, striatal, thalamic. Regions that
landed next to each other did so because their lipid profiles are similar, which is the clustering
finding structure with no anatomy labels used in the sorting at all.

## 9. why this anatomy powers the next notebooks

Step back and see what we built, and why it matters downstream.

- We loaded the per-pixel CCF coordinates that registration (ABBA, refined in the original atlas by
  STalign) produced, the one piece of infrastructure shipped ready-made, and made the coordinate-to-region
  lookup explicit.
- We drew region maps for control and pregnant in the same atlas frame, so the two sections are directly
  comparable region by region.
- We split grey from white matter straight from the Allen fiber-tract acronyms, and confirmed the myelin
  lipid HexCer 42:2 traces the white-matter tracts on its own, the first proof that anatomy and lipidomics
  agree.
- We built the region-by-lipid matrix, the mean lipid profile of every region.

That region-by-lipid matrix is the hinge for what comes next:

- region-level differential lipidomics. With both sections in the same atlas, we can ask, for each region,
  which lipids changed between control and pregnant, region by region.
- the gene join. Gene-expression atlases can produce expression averaged over the same Allen regions, and the
  Allen acronym is the shared key: join our region-by-lipid table to a region-by-gene table on `acronym`,
  and we can finally ask how a region's lipid profile and its gene-expression profile are related.

You now know what a reference atlas is, how registration (affine plus nonlinear warp, done by ABBA in
practice) puts every pixel on it, how a coordinate indexes the annotation volume, and how to turn pixels
into region-level views. The next notebook puts this to work on the biology: which lipids change, region
by region, in pregnancy.